In [1]:
import os
import random

VAL_DIR = "/kaggle/input/datasets/ifigotin/imagenetmini-1000/imagenet-mini/val"
NUM_IMAGES = 200   # small for now, we increase later
SEED = 42

# 1) All 1000 class folders, sorted.
#    Sorted folder order = official ImageNet class index order (0 to 999)
synsets = sorted(
    folder for folder in os.listdir(VAL_DIR)
    if os.path.isdir(os.path.join(VAL_DIR, folder))
)

synset_to_index = {name: i for i, name in enumerate(synsets)}

print("Number of classes:", len(synsets))

# 2) Collect every validation image with its true class folder
all_images = []

for synset in synsets:
    folder = os.path.join(VAL_DIR, synset)
    for filename in sorted(os.listdir(folder)):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            all_images.append(
                (os.path.join(folder, filename), synset)
            )

print("Total validation images:", len(all_images))

# 3) Same random sample in every notebook (because SEED is fixed)
random.seed(SEED)
eval_images = random.sample(all_images, NUM_IMAGES)

print("Images selected for evaluation:", len(eval_images))

Number of classes: 1000
Total validation images: 3923
Images selected for evaluation: 200


In [2]:

# Verify the mapping is correct
print("index 0   ->", synsets[0])
print("index 11  ->", synsets[11])
print("index 999 ->", synsets[999])

# Show first 3 evaluation samples
for path, synset in eval_images[:3]:
    print(os.path.basename(path), "| true class:", synset, "| index:", synset_to_index[synset])

index 0   -> n01440764
index 11  -> n01531178
index 999 -> n15075141
ILSVRC2012_val_00049034.JPEG | true class: n03777568 | index: 661
ILSVRC2012_val_00030908.JPEG | true class: n02011460 | index: 133
ILSVRC2012_val_00028352.JPEG | true class: n01641577 | index: 30


In [3]:
from transformers import CLIPProcessor, CLIPModel

print("CLIP libraries imported successfully!")

CLIP libraries imported successfully!


In [4]:
clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
)

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

print("CLIP model loaded successfully!")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

... (3 outputs left out)

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP model loaded successfully!


In [5]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: cpu


In [6]:
clip_model = clip_model.to(device)

print("CLIP using device:", device)

CLIP using device: cpu


In [7]:
import os
from tensorflow.keras.preprocessing import image

# ImageNet Mini dataset folder
dataset_folder = "/kaggle/input/datasets/ifigotin/imagenetmini-1000"

# Find image files again
image_files = []

for root, dirs, files in os.walk(dataset_folder):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
            image_files.append(
                os.path.join(root, file)
            )

print("Total images found:", len(image_files))

# Select the first image
image_path = image_files[0]

# Load image
img = image.load_img(
    image_path,
    target_size=(224, 224)
)

print("Testing image:", os.path.basename(image_path))


Total images found: 38668
Testing image: ILSVRC2012_val_00029581.JPEG


In [8]:
labels = [
    "a photo of a goldfinch",
    "a photo of a dog",
    "a photo of a cat",
    "a photo of a car",
    "a photo of a bird"
]

print("Labels:")
for label in labels:
    print("-", label)

Labels:
- a photo of a goldfinch
- a photo of a dog
- a photo of a cat
- a photo of a car
- a photo of a bird


In [9]:
inputs = clip_processor(
    text=labels,
    images=img,
    return_tensors="pt",
    padding=True
)

# Move everything to the same device as CLIP
inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

print("Image and text successfully prepared for CLIP!")

Image and text successfully prepared for CLIP!


In [10]:
with torch.no_grad():
    outputs = clip_model(**inputs)

# Get similarity scores
logits_per_image = outputs.logits_per_image

# Convert scores to probabilities
probabilities = torch.softmax(
    logits_per_image,
    dim=1
)[0]

# Sort predictions from highest to lowest
sorted_indices = torch.argsort(
    probabilities,
    descending=True
)

print("\nCLIP Zero-Shot Predictions:")
print("-" * 40)

for i, index in enumerate(sorted_indices, start=1):

    label = labels[index]
    confidence = probabilities[index].item()

    print(
        f"{i}. {label} - "
        f"{confidence * 100:.2f}%"
    )


CLIP Zero-Shot Predictions:
----------------------------------------
1. a photo of a bird - 91.98%
2. a photo of a goldfinch - 8.01%
3. a photo of a dog - 0.01%
4. a photo of a car - 0.00%
5. a photo of a cat - 0.00%


In [11]:
import os

dataset_folder = "/kaggle/input/datasets/ifigotin/imagenetmini-1000"

for root, dirs, files in os.walk(dataset_folder):
    print(root)
    print("Subfolders:", dirs[:5])
    print("Files:", files[:3])
    print("-" * 60)

    if root.count(os.sep) >= dataset_folder.count(os.sep) + 3:
        break

/kaggle/input/datasets/ifigotin/imagenetmini-1000
Subfolders: ['imagenet-mini']
Files: []
------------------------------------------------------------
/kaggle/input/datasets/ifigotin/imagenetmini-1000/imagenet-mini
Subfolders: ['val', 'train']
Files: []
------------------------------------------------------------
/kaggle/input/datasets/ifigotin/imagenetmini-1000/imagenet-mini/val
Subfolders: ['n01531178', 'n02412080', 'n02098413', 'n01797886', 'n04589890']
Files: []
------------------------------------------------------------
/kaggle/input/datasets/ifigotin/imagenetmini-1000/imagenet-mini/val/n01531178
Subfolders: []
Files: ['ILSVRC2012_val_00029581.JPEG', 'ILSVRC2012_val_00048710.JPEG', 'ILSVRC2012_val_00001274.JPEG']
------------------------------------------------------------


In [12]:
import os

# Validation folder
val_folder = "/kaggle/input/datasets/ifigotin/imagenetmini-1000/imagenet-mini/val"

# Get class folders
class_folders = [
    folder for folder in os.listdir(val_folder)
    if os.path.isdir(os.path.join(val_folder, folder))
]

print("Number of classes:", len(class_folders))

# Count validation images
total_val_images = 0

for class_folder in class_folders:
    class_path = os.path.join(val_folder, class_folder)

    for file in os.listdir(class_path):
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            total_val_images += 1

print("Total validation images:", total_val_images)

Number of classes: 1000
Total validation images: 3923


In [13]:
import os

val_folder = "/kaggle/input/datasets/ifigotin/imagenetmini-1000/imagenet-mini/val"

image_paths = []
true_labels = []

# Go through every class folder
for class_name in sorted(os.listdir(val_folder)):

    class_path = os.path.join(
        val_folder,
        class_name
    )

    # Make sure it is a folder
    if not os.path.isdir(class_path):
        continue

    # Go through images inside the class folder
    for filename in os.listdir(class_path):

        if filename.lower().endswith(
            (".jpg", ".jpeg", ".png")
        ):

            image_path = os.path.join(
                class_path,
                filename
            )

            image_paths.append(image_path)
            true_labels.append(class_name)

print("Total images:", len(image_paths))
print("Total labels:", len(true_labels))

print("\nFirst 5 examples:")

for i in range(5):
    print(
        os.path.basename(image_paths[i]),
        "→",
        true_labels[i]
    )

Total images: 3923
Total labels: 3923

First 5 examples:
ILSVRC2012_val_00030740.JPEG → n01440764
ILSVRC2012_val_00046252.JPEG → n01440764
ILSVRC2012_val_00009111.JPEG → n01440764
ILSVRC2012_val_00020822.JPEG → n01443537
ILSVRC2012_val_00000994.JPEG → n01443537


In [14]:
# Look at some ViT ImageNet class mappings

for class_id in range(5):
    print(
        class_id,
        "→",
        vit_model.config.id2label[class_id]
    )

NameError: name 'vit_model' is not defined

In [ ]:
from transformers import ViTImageProcessor, ViTForImageClassification

print("Loading ViT...")

vit_processor = ViTImageProcessor.from_pretrained(
    "google/vit-base-patch16-224"
)

vit_model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224"
)

print("ViT loaded successfully!")

In [ ]:
print("Number of ViT classes:",
      len(vit_model.config.id2label))

print("\nFirst 10 ViT labels:")

for class_id in range(10):
    print(
        class_id,
        "→",
        vit_model.config.id2label[class_id]
    )

In [ ]:
import os

dataset_folder = "/kaggle/input/datasets/ifigotin/imagenetmini-1000"

print("Files in dataset:")
print()

for root, dirs, files in os.walk(dataset_folder):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
print("Example model labels:")

for i in range(20):
    print(i, "→", vit_model.config.id2label[i])

In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"

urllib.request.urlretrieve(
    url,
    "/kaggle/working/imagenet_classes.txt"
)

print("ImageNet class list downloaded successfully!")

In [ ]:
with open("/kaggle/working/imagenet_classes.txt", "r") as f:
    imagenet_classes = [line.strip() for line in f.readlines()]

print("Number of ImageNet classes:", len(imagenet_classes))

print("\nFirst 20 classes:")

for i in range(20):
    print(i, "→", imagenet_classes[i])

In [ ]:
import torchvision

print("Torchvision version:", torchvision.__version__)

# Check whether ImageNet class metadata is available
from torchvision.models import ResNet50_Weights

weights = ResNet50_Weights.DEFAULT

print("Number of ImageNet classes:", len(weights.meta["categories"]))
print("\nFirst 20 classes:")

for i, name in enumerate(weights.meta["categories"][:20]):
    print(i, "→", name)

In [ ]:
from tensorflow.keras.applications.imagenet_utils import CLASS_INDEX

print("Number of classes:", len(CLASS_INDEX))

print("\nFirst 5 mappings:")

for i in range(5):
    synset, name = CLASS_INDEX[str(i)]
    print(i, "→", synset, "→", name)

In [ ]:
from torchvision.models import ResNet50_Weights

weights = ResNet50_Weights.DEFAULT

print("Available metadata keys:")
print(weights.meta.keys())

In [ ]:
import torch
from torchvision.models import ResNet50_Weights

weights = ResNet50_Weights.DEFAULT
categories = weights.meta["categories"]

print("Number of ImageNet classes:", len(categories))

for i in range(10):
    print(i, "→", categories[i])

In [ ]:
import os
import time
import torch
import pandas as pd
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

clip_model = clip_model.to(device)
clip_model.eval()

print("CLIP ready on:", device)

In [ ]:
from transformers import ViTForImageClassification

# Borrow ViT's label list (same index order as our sorted synsets)
vit_config_model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224")

class_names = []
for i in range(1000):
    full_label = vit_config_model.config.id2label[i]
    # Labels look like "bullfrog, Rana catesbeiana" -> keep only the first name
    short_name = full_label.split(",")[0].strip()
    class_names.append(short_name)

index_to_synset = {i: name for i, name in enumerate(synsets)}

print("Total class names:", len(class_names))
print("Example 0  :", class_names[0])
print("Example 11 :", class_names[11])
print("Example 999:", class_names[999])

In [ ]:
prompts = [f"a photo of a {name}" for name in class_names]

with torch.no_grad():
    text_inputs = clip_processor(
        text=prompts,
        return_tensors="pt",
        padding=True
    ).to(device)

    text_output = clip_model.get_text_features(**text_inputs)
    text_features = extract_features(text_output)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

print("Text features shape:", text_features.shape)

In [ ]:
dummy = Image.new("RGB", (224, 224))
dummy_inputs = clip_processor(images=dummy, return_tensors="pt").to(device)

with torch.no_grad():
    dummy_out = clip_model.get_image_features(**dummy_inputs)
    dummy_feat = extract_features(dummy_out)

print("Warm-up done. Image feature shape:", dummy_feat.shape)

In [ ]:
test_img_inputs = clip_processor(
    images=Image.new("RGB", (224, 224)),
    return_tensors="pt"
).to(device)

test_txt_inputs = clip_processor(
    text=["a photo of a dog"],
    return_tensors="pt",
    padding=True
).to(device)

with torch.no_grad():
    img_out = clip_model.get_image_features(**test_img_inputs)
    txt_out = clip_model.get_text_features(**test_txt_inputs)

print("IMAGE return type:", type(img_out).__name__)
if not torch.is_tensor(img_out):
    for key in img_out.keys():
        value = img_out[key]
        if torch.is_tensor(value):
            print("   ", key, tuple(value.shape))
else:
    print("    tensor shape:", tuple(img_out.shape))

print()
print("TEXT return type:", type(txt_out).__name__)
if not torch.is_tensor(txt_out):
    for key in txt_out.keys():
        value = txt_out[key]
        if torch.is_tensor(value):
            print("   ", key, tuple(value.shape))
else:
    print("    tensor shape:", tuple(txt_out.shape))

print()
print("visual_projection:", clip_model.visual_projection.in_features, "->", clip_model.visual_projection.out_features)
print("text_projection  :", clip_model.text_projection.in_features, "->", clip_model.text_projection.out_features)

In [ ]:
def extract_features(output):
    """Return CLIP's final 512-dim features, whatever the version gives back."""
    if torch.is_tensor(output):
        return output
    return output.pooler_output

print("Helper updated.")

In [ ]:
results = []

for count, (image_path, true_synset) in enumerate(eval_images, start=1):

    img = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=img, return_tensors="pt").to(device)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.time()

    with torch.no_grad():
        image_output = clip_model.get_image_features(**inputs)
        image_features = extract_features(image_output)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        # Compare this image against all 1000 prompts
        similarity = image_features @ text_features.T
        probabilities = torch.softmax(similarity * 100.0, dim=-1)[0]

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.time()

    inference_time = end_time - start_time

    predicted_index = int(torch.argmax(probabilities))
    confidence = float(probabilities[predicted_index])

    results.append({
        "model": "CLIP",
        "image_name": os.path.basename(image_path),
        "image_path": image_path,
        "true_synset": true_synset,
        "predicted_synset": index_to_synset[predicted_index],
        "predicted_label": class_names[predicted_index],
        "confidence": confidence,
        "correct": int(index_to_synset[predicted_index] == true_synset),
        "inference_time": inference_time,
    })

    if count % 50 == 0:
        print("Processed", count, "images")

print("Done. Total results:", len(results))

In [ ]:
df = pd.DataFrame(results)

df.to_csv("/kaggle/working/clip_results.csv", index=False)

print("===== CLIP (Zero-Shot) Summary =====")
print("Images tested        :", len(df))
print("Top-1 Accuracy       :", round(df["correct"].mean() * 100, 2), "%")
print("Avg inference time   :", round(df["inference_time"].mean() * 1000, 2), "ms")
print("Images per second    :", round(1 / df["inference_time"].mean(), 2))

df.head()

In [ ]:
import os
print(os.listdir("/kaggle/working"))

In [ ]:
import os
import json
from transformers import ViTForImageClassification

VAL_DIR = "/kaggle/input/datasets/ifigotin/imagenetmini-1000/imagenet-mini/val"

synsets = sorted(
    folder for folder in os.listdir(VAL_DIR)
    if os.path.isdir(os.path.join(VAL_DIR, folder))
)

vit_cfg = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224")

class_names = [
    vit_cfg.config.id2label[i].split(",")[0].strip()
    for i in range(1000)
]

data = {"synsets": synsets, "class_names": class_names}

with open("/kaggle/working/imagenet_labels.json", "w") as f:
    json.dump(data, f)

print("Saved", len(class_names), "class names")
print("Check:", class_names[0], "|", class_names[11], "|", class_names[999])

In [ ]:
from IPython.display import FileLink
FileLink("imagenet_labels.json")